<div dir="rtl" align="right">

# كشفُ الآثارِ الشائبةِ بِـ KNN

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نَستخدمُ خوارزميّةَ الجيرانِ الأقربينَ (KNN) لِكشفِ الآثارِ الشائبةِ في إشارةِ EEG. نَستخرجُ خصائصَ إحصائيّةً من نوافذَ زمنيّةٍ ونُدرّبُ النموذجَ على التمييزِ بينَ النوافذِ النظيفةِ والمشوبة.

## المُخرجاتُ المُتوقّعةُ

- الآثارُ الحقيقيّةُ المُحدّدةُ بِالعتبةِ في الأعلى
- توقّعاتُ KNN في الأسفل معَ نسبةِ الدقّة
- مناطقُ مُظلّلةٌ بالأحمرِ تُشيرُ إلى الآثارِ المُكتشفةِ

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| القناةُ | P4 | المنطقةُ الجداريةُ |
| WINDOW_SIZE | 200 | نافذةُ ثانيةٍ واحدةٍ |
| n_neighbors | 5 | عددُ الجيرانِ |
| threshold | 3*std | عتبةُ الآثارِ |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install mne scikit-learn EMD-signal scipy numpy plotly wfdb


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2.

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. تطبيقُ KNN لِكشفِ الآثارِ

نَستخرجُ خصائصَ (تباين، تَفلطح، سعةٌ قصوى) من كلِّ نافذةٍ، ونُسمّيها بناءً على عتبةِ السعة، ثمّ نُدرّبُ KNN.

</div>

In [ ]:
from scipy.stats import kurtosis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

channel_data = eeg_data[:, 0]
WINDOW_SIZE = 200
n_windows = len(channel_data) // WINDOW_SIZE
windows = channel_data[:n_windows * WINDOW_SIZE].reshape(n_windows, WINDOW_SIZE)

features = np.column_stack([
    np.var(windows, axis=1),
    kurtosis(windows, axis=1),
    np.max(np.abs(windows), axis=1),
])

threshold = 3 * np.std(channel_data)
labels = (np.max(np.abs(windows), axis=1) > threshold).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.3, random_state=42
)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
predictions = knn.predict(features)
accuracy = accuracy_score(y_test, knn.predict(X_test))
print(f'Accuracy: {accuracy:.1%}')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- المناطقُ المُظلّلةُ بالأحمرِ تُشيرُ إلى الآثارِ
- الأعلى: الآثارُ الحقيقيّةُ، الأسفل: توقّعاتُ KNN
- استخدمْ التكبيرَ لِفحصِ نطاقاتٍ زمنيةٍ مُحدّدةٍ


</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

t_sec = np.arange(len(channel_data)) / fs

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=(f'Ground truth (threshold={threshold:.0f} uV)',
                                    f'KNN predictions (accuracy={accuracy:.1%})'))
fig.add_trace(go.Scatter(x=t_sec, y=channel_data, name='Signal',
                         line=dict(color='blue', width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=channel_data, name='Signal',
                         line=dict(color='blue', width=0.5)), row=2, col=1)

for i in range(n_windows):
    t_start = i * WINDOW_SIZE / fs
    t_end = (i + 1) * WINDOW_SIZE / fs
    if labels[i] == 1:
        fig.add_vrect(x0=t_start, x1=t_end, fillcolor='red', opacity=0.3,
                      line_width=0, row=1, col=1)
    if predictions[i] == 1:
        fig.add_vrect(x0=t_start, x1=t_end, fillcolor='red', opacity=0.3,
                      line_width=0, row=2, col=1)

fig.update_layout(height=700, title_text='KNN Artifact Detection - Channel P4',
                  xaxis2_title='Time (s)', showlegend=False)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- KNN يُصنّفُ النوافذَ بناءً على تشابهِها معَ الجيرانِ
- الخصائصُ الإحصائيّةُ تَكفي لِكشفِ الآثارِ ذاتِ السعةِ العالية
- جودةُ التسميةِ تُحدّدُ جودةَ النموذج
- يَحتاجُ إلى تسميةٍ يدويّةٍ من خبيرٍ في الممارسةِ العمليّة


</div>